In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta


directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2021/'

# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

# monthly data
monthly_data = []

year = 2021

 # Loop monthly
for month in range(1, 13): 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:  # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
            # Gå i stå, slet tomme filer
            if os.path.getsize(message_file) == 0 or os.path.getsize(orderbook_file) == 0:
                print(f"Skipping empty file: {message_file} or {orderbook_file}")
                continue  

            try:
        
                message_df = pd.read_csv(message_file, encoding='utf-8', low_memory=False)
                orderbook_df = pd.read_csv(orderbook_file, encoding='utf-8', low_memory=False)
            except Exception as e:
                print(f"Error loading file {filename}: {e}")
                continue  # Skip this file if there's an error

            # Dropper col 7
            message_df = message_df.iloc[:, :-1]  # Drop last column
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            # Start dag SPY_2021-01-04
            base_date = filename.split('_')[1]
            
            
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merge message and orderbook 
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Slet NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' as index for 1 sec
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            
            monthly_files.append(resampled_df)
    
    # Concatenate all daily files for the month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)


final_df = pd.concat(monthly_data)


final_df.to_csv(f'combined_SPY_{year}_cleaned.csv', index=False)

print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2021-01-04 09:30:00    3753100.0       700.0    3752800.0       100.0   
2021-01-04 09:30:01    3753300.0      1792.0    3753000.0       200.0   
2021-01-04 09:30:02    3752800.0        20.0    3752500.0       102.0   
2021-01-04 09:30:03    3753500.0      1000.0    3753200.0       367.0   
2021-01-04 09:30:04    3753600.0       800.0    3753300.0       187.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2021-01-04 09:30:00    3753300.0      1500.0    3752600.0       100.0   
2021-01-04 09:30:01    3753400.0       600.0    3752900.0       200.0   
2021-01-04 09:30:02    3752900.0       100.0    3752400.0       100.0   
2021-01-04 09:30:03    3753600.0      1900.0    3753100.0       100.0   
2021-01-04 09:30:04    3753700.0      1100.0    37

In [2]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2021-12-31 15:59:55    4748400.0      3500.0    4748200.0       300.0   
2021-12-31 15:59:56    4747700.0      1200.0    4747500.0       400.0   
2021-12-31 15:59:57    4747400.0       200.0    4747300.0       400.0   
2021-12-31 15:59:58    4747700.0      1100.0    4747600.0       100.0   
2021-12-31 15:59:59    4747200.0       100.0    4747100.0      1100.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2021-12-31 15:59:55    4748500.0      3100.0    4748100.0       488.0   
2021-12-31 15:59:56    4747800.0      2300.0    4747400.0       200.0   
2021-12-31 15:59:57    4747500.0      3200.0    4747200.0       500.0   
2021-12-31 15:59:58    4747800.0       500.0    4747500.0       400.0   
2021-12-31 15:59:59    4747300.0       200.0    47

In [3]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5602633


In [4]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2021-01-04 09:30:00,3753100.0,700.0,3752800.0,100.0,3753300.0,1500.0,3752600.0,100.0,4.0,33235904.0,600.0,3753100.0,-1.0
2021-01-04 09:30:01,3753300.0,1792.0,3753000.0,200.0,3753400.0,600.0,3752900.0,200.0,3.0,33574124.0,100.0,3753100.0,1.0
2021-01-04 09:30:02,3752800.0,20.0,3752500.0,102.0,3752900.0,100.0,3752400.0,100.0,3.0,33874448.0,1000.0,3752800.0,-1.0
2021-01-04 09:30:03,3753500.0,1000.0,3753200.0,367.0,3753600.0,1900.0,3753100.0,100.0,1.0,34068152.0,1000.0,3753500.0,-1.0
2021-01-04 09:30:04,3753600.0,800.0,3753300.0,187.0,3753700.0,1100.0,3753200.0,122.0,1.0,34237376.0,100.0,3753200.0,1.0


In [5]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2021-12-31 15:59:55,4748400.0,3500.0,4748200.0,300.0,4748500.0,3100.0,4748100.0,488.0,1.0,591008672.0,1000.0,4748400.0,-1.0
2021-12-31 15:59:56,4747700.0,1200.0,4747500.0,400.0,4747800.0,2300.0,4747400.0,200.0,1.0,591145604.0,200.0,4747500.0,1.0
2021-12-31 15:59:57,4747400.0,200.0,4747300.0,400.0,4747500.0,3200.0,4747200.0,500.0,1.0,591272476.0,200.0,4747300.0,1.0
2021-12-31 15:59:58,4747700.0,1100.0,4747600.0,100.0,4747800.0,500.0,4747500.0,400.0,3.0,591416628.0,100.0,4747600.0,1.0
2021-12-31 15:59:59,4747200.0,100.0,4747100.0,1100.0,4747300.0,200.0,4747000.0,800.0,1.0,591562052.0,100.0,4747200.0,-1.0


In [6]:
# # Save 
# final_df.to_csv('final_combined_2021.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2021_with_time.csv', index=True)

In [7]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 251


In [8]:
# Load the data from 'final_combined_2021_with_time.csv'
final_combined_2021_with_time = pd.read_csv('final_combined_2021_with_time.csv')


final_combined_2021_with_time['Time (sec)'] = pd.to_datetime(final_combined_2021_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2021_with_time.groupby(final_combined_2021_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate through each group each day
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [9]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2021-01-04 09:30:00          3753100.0        3754500.0        3745200.0   
1 2021-01-04 09:35:00          3745500.0        3745800.0        3739000.0   
2 2021-01-04 09:40:00          3740000.0        3741200.0        3735600.0   
3 2021-01-04 09:45:00          3741200.0        3743300.0        3740000.0   
4 2021-01-04 09:50:00          3742000.0        3742000.0        3731200.0   
5 2021-01-04 09:55:00          3736200.0        3737300.0        3732900.0   
6 2021-01-04 10:00:00          3735200.0        3737300.0        3731100.0   
7 2021-01-04 10:05:00          3732500.0        3733800.0        3727800.0   
8 2021-01-04 10:10:00          3729000.0        3730000.0        3725800.0   
9 2021-01-04 10:15:00          3725900.0        3727800.0        3723500.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3745400.0          3752800.0        3754200.0        37

In [10]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2021_corrected.csv', index=False)

In [11]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2021-01-04    58
2021-01-05    78
2021-01-06    78
2021-01-07    78
2021-01-08    78
              ..
2021-12-27    78
2021-12-28    78
2021-12-29    78
2021-12-30    78
2021-12-31    78
Length: 251, dtype: int64


In [12]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2021_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2021-01-04 09:30:00          3753100.0        3754500.0        3745200.0   
1 2021-01-04 09:35:00          3745500.0        3745800.0        3739000.0   
2 2021-01-04 09:40:00          3740000.0        3741200.0        3735600.0   
3 2021-01-04 09:45:00          3741200.0        3743300.0        3740000.0   
4 2021-01-04 09:50:00          3742000.0        3742000.0        3731200.0   
5 2021-01-04 09:55:00          3736200.0        3737300.0        3732900.0   
6 2021-01-04 10:00:00          3735200.0        3737300.0        3731100.0   
7 2021-01-04 10:05:00          3732500.0        3733800.0        3727800.0   
8 2021-01-04 10:10:00          3729000.0        3730000.0        3725800.0   
9 2021-01-04 10:15:00          3725900.0        3727800.0        3723500.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3745400.0          3752800.0        3754200.0        37

In [13]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2021_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2021-01-04 09:30:00,3753100.0,3754500.0,3745200.0,3745400.0,3752800.0,3754200.0,3745000.0,3745300.0,3753300.0,3754600.0,3745300.0,3745500.0,3752600.0,3754100.0,3744900.0,3745200.0,163547.0,545.156667,121275.0,404.250000,264811.0,882.703333,235185.0,783.950000,3753100.0,3754400.0,3745000.0,3745300.0,0.053333,2021-01-04
2021-01-04 09:35:00,3745500.0,3745800.0,3739000.0,3740300.0,3745300.0,3745700.0,3738800.0,3740200.0,3745600.0,3745900.0,3739100.0,3740400.0,3745200.0,3745600.0,3738700.0,3740100.0,101272.0,337.573333,160863.0,536.210000,205335.0,684.450000,272256.0,907.520000,3745500.0,3745700.0,3738800.0,3740400.0,-0.073333,2021-01-04
2021-01-04 09:40:00,3740000.0,3741200.0,3735600.0,3741200.0,3739800.0,3741000.0,3735500.0,3741000.0,3740100.0,3741300.0,3735700.0,3741300.0,3739700.0,3740900.0,3735400.0,3740900.0,159042.0,530.140000,126878.0,422.926667,253006.0,843.353333,206785.0,689.283333,3740000.0,3741200.0,3735500.0,3741100.0,0.126667,2021-01-04
2021-01-04 09:45:00,3741200.0,3743300.0,3740000.0,3742100.0,3741000.0,3743100.0,3739800.0,3742000.0,3741300.0,3743400.0,3740100.0,3742200.0,3740900.0,3743000.0,3739700.0,3741900.0,161916.0,539.720000,134322.0,447.740000,303079.0,1010.263333,256716.0,855.720000,3741000.0,3743150.0,3739800.0,3742100.0,0.146667,2021-01-04
2021-01-04 09:50:00,3742000.0,3742000.0,3731200.0,3736200.0,3741800.0,3741800.0,3731100.0,3736000.0,3742100.0,3742100.0,3731300.0,3736300.0,3741700.0,3741700.0,3731000.0,3735900.0,144719.0,482.396667,128870.0,429.566667,261886.0,872.953333,272324.0,907.746667,3741800.0,3741800.0,3731200.0,3736200.0,0.173333,2021-01-04
2021-01-04 09:55:00,3736200.0,3737300.0,3732900.0,3734100.0,3736100.0,3737100.0,3732800.0,3734000.0,3736300.0,3737400.0,3733000.0,3734200.0,3736000.0,3737000.0,3732700.0,3733900.0,137190.0,457.300000,100505.0,335.016667,241872.0,806.240000,245773.0,819.243333,3736300.0,3737300.0,3732800.0,3734100.0,0.046667,2021-01-04
2021-01-04 10:00:00,3735200.0,3737300.0,3731100.0,3732800.0,3735100.0,3737100.0,3730900.0,3732600.0,3735300.0,3737400.0,3731200.0,3732900.0,3735000.0,3737000.0,3730800.0,3732500.0,90231.0,300.770000,85531.0,285.103333,207674.0,692.246667,204479.0,681.596667,3735100.0,3737300.0,3730900.0,3732700.0,0.033333,2021-01-04
2021-01-04 10:05:00,3732500.0,3733800.0,3727800.0,3729100.0,3732300.0,3733600.0,3727600.0,3728900.0,3732600.0,3733900.0,3727900.0,3729200.0,3732200.0,3733500.0,3727500.0,3728800.0,78175.0,260.583333,96094.0,320.313333,184871.0,616.236667,199192.0,663.973333,3732300.0,3733600.0,3727600.0,3728900.0,0.206667,2021-01-04
2021-01-04 10:10:00,3729000.0,3730000.0,3725800.0,3725900.0,3728800.0,3729800.0,3725600.0,3725700.0,3729100.0,3730100.0,3725900.0,3726000.0,3728700.0,3729700.0,3725500.0,3725600.0,83063.0,276.876667,95900.0,319.666667,222288.0,740.960000,244433.0,814.776667,3728800.0,3730000.0,3725600.0,3725600.0,0.040000,2021-01-04
2021-01-04 10:15:00,3725900.0,3727800.0,3723500.0,3724800.0,3725700.0,3727600.0,3723300.0,3724600.0,3726000.0,3727900.0,3723600.0,3724900.0,3725600.0,3727500.0,3723200.0,3724500.0,83522.0,278.406667,100128.0,333.760000,262641.0,875.470000,256799.0,855.996667,3725800.0,3727800.0,3723400.0,3724900.0,-0.113333,2021-01-04


In [14]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.tail(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2021_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2021-12-31 15:10:00,4762000.0,4763100.0,4759500.0,4759800.0,4761900.0,4763000.0,4759300.0,4759700.0,4762100.0,4763200.0,4759600.0,4759900.0,4761800.0,4762900.0,4759200.0,4759600.0,158540.0,528.466667,272759.0,909.196667,245678.0,818.926667,258262.0,860.873333,4762000.0,4763100.0,4759300.0,4759700.0,-0.073333,2021-12-31
2021-12-31 15:15:00,4759800.0,4762300.0,4758800.0,4762300.0,4759700.0,4762200.0,4758700.0,4762200.0,4759900.0,4762400.0,4758900.0,4762400.0,4759600.0,4762100.0,4758600.0,4762100.0,120876.0,402.920000,212973.0,709.910000,207248.0,690.826667,224687.0,748.956667,4759700.0,4762300.0,4758700.0,4762100.0,0.040000,2021-12-31
2021-12-31 15:20:00,4762300.0,4762300.0,4759500.0,4761700.0,4762200.0,4762200.0,4759400.0,4761600.0,4762400.0,4762400.0,4759600.0,4761800.0,4762100.0,4762100.0,4759300.0,4761500.0,149749.0,499.163333,269682.0,898.940000,332873.0,1109.576667,310837.0,1036.123333,4762200.0,4762200.0,4759500.0,4761700.0,-0.093333,2021-12-31
2021-12-31 15:25:00,4761700.0,4765500.0,4761200.0,4764500.0,4761600.0,4765400.0,4761100.0,4764400.0,4761800.0,4765600.0,4761300.0,4764600.0,4761500.0,4765300.0,4761000.0,4764300.0,128671.0,428.903333,300555.0,1001.850000,196185.0,653.950000,330374.0,1101.246667,4761600.0,4765500.0,4761100.0,4764500.0,0.060000,2021-12-31
2021-12-31 15:30:00,4764500.0,4766100.0,4760300.0,4760700.0,4764400.0,4766000.0,4760200.0,4760600.0,4764600.0,4766200.0,4760400.0,4760800.0,4764300.0,4765900.0,4760100.0,4760500.0,134615.0,448.716667,174304.0,581.013333,223045.0,743.483333,216463.0,721.543333,4764400.0,4766000.0,4760300.0,4760600.0,0.006667,2021-12-31
2021-12-31 15:35:00,4760700.0,4761800.0,4759700.0,4761800.0,4760500.0,4761700.0,4759600.0,4761700.0,4760800.0,4761900.0,4759800.0,4761900.0,4760400.0,4761600.0,4759500.0,4761600.0,133378.0,444.593333,223199.0,743.996667,184290.0,614.300000,244689.0,815.630000,4760600.0,4761700.0,4759600.0,4761700.0,0.046667,2021-12-31
2021-12-31 15:40:00,4761800.0,4764500.0,4761200.0,4764200.0,4761700.0,4764400.0,4761100.0,4764100.0,4761900.0,4764600.0,4761300.0,4764300.0,4761600.0,4764300.0,4761000.0,4764000.0,130688.0,435.626667,309017.0,1030.056667,196233.0,654.110000,210493.0,701.643333,4761700.0,4764500.0,4761100.0,4764100.0,-0.013333,2021-12-31
2021-12-31 15:45:00,4763900.0,4766600.0,4762000.0,4762000.0,4763800.0,4766400.0,4761800.0,4761800.0,4764000.0,4766700.0,4762100.0,4762100.0,4763700.0,4766300.0,4761700.0,4761700.0,116721.0,389.070000,234185.0,780.616667,238222.0,794.073333,254854.0,849.513333,4764000.0,4766500.0,4761800.0,4762000.0,-0.006667,2021-12-31
2021-12-31 15:50:00,4761900.0,4761900.0,4749500.0,4750000.0,4761800.0,4761800.0,4749400.0,4749900.0,4762000.0,4762000.0,4749600.0,4750100.0,4761700.0,4761700.0,4749300.0,4749800.0,155983.0,519.943333,156449.0,521.496667,192812.0,642.706667,205582.0,685.273333,4761800.0,4761800.0,4749400.0,4749800.0,-0.020000,2021-12-31
2021-12-31 15:55:00,4750000.0,4751600.0,4747000.0,4747200.0,4749800.0,4751500.0,4746800.0,4747100.0,4750100.0,4751700.0,4747100.0,4747300.0,4749700.0,4751400.0,4746700.0,4747000.0,286896.0,956.320000,233662.0,778.873333,372861.0,1242.870000,254336.0,847.786667,4749900.0,4751500.0,4746900.0,4747200.0,-0.013333,2021-12-31
